In [1]:
import torch
from vit_pytorch.efficient import ViT
from linformer import Linformer

# Define transformer
transformer = Linformer(
    dim=90,             # changed from 80 to 90
    seq_len=50,     
    depth=6,
    heads=15,
    k=64,
    dropout=0.2
)

# Define ViT model
num_channels = 4  # example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ViT(
    dim=90,           # changed from 80 to 90
    image_size=224,
    patch_size=32,
    num_classes=3,
    transformer=transformer,
    channels=num_channels
).to(device)

# Print all learnable parameters with shapes
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.shape}")

# Count total learnable parameters
total_vit_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_vit_params}")


pos_embedding: torch.Size([1, 50, 90])
cls_token: torch.Size([1, 1, 90])
to_patch_embedding.1.weight: torch.Size([4096])
to_patch_embedding.1.bias: torch.Size([4096])
to_patch_embedding.2.weight: torch.Size([90, 4096])
to_patch_embedding.2.bias: torch.Size([90])
to_patch_embedding.3.weight: torch.Size([90])
to_patch_embedding.3.bias: torch.Size([90])
transformer.net.layers.0.0.fn.proj_k: torch.Size([50, 64])
transformer.net.layers.0.0.fn.proj_v: torch.Size([50, 64])
transformer.net.layers.0.0.fn.to_q.weight: torch.Size([90, 90])
transformer.net.layers.0.0.fn.to_k.weight: torch.Size([90, 90])
transformer.net.layers.0.0.fn.to_v.weight: torch.Size([90, 90])
transformer.net.layers.0.0.fn.to_out.weight: torch.Size([90, 90])
transformer.net.layers.0.0.fn.to_out.bias: torch.Size([90])
transformer.net.layers.0.0.norm.weight: torch.Size([90])
transformer.net.layers.0.0.norm.bias: torch.Size([90])
transformer.net.layers.0.1.fn.w1.weight: torch.Size([360, 90])
transformer.net.layers.0.1.fn.w1.bia

In [8]:
from gwtorch.modules.neural_net import CNN_Model

# Instantiate model
model = CNN_Model(input_channels=4)

# Print learnable parameters with their shapes
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.shape}")

# Count total learnable parameters
total_cnn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_cnn_params}")

conv1.weight: torch.Size([32, 4, 3, 3])
conv1.bias: torch.Size([32])
conv2.weight: torch.Size([64, 32, 3, 3])
conv2.bias: torch.Size([64])
conv3.weight: torch.Size([128, 64, 3, 3])
conv3.bias: torch.Size([128])
fc1.weight: torch.Size([256, 100352])
fc1.bias: torch.Size([256])
fc2.weight: torch.Size([3, 256])
fc2.bias: torch.Size([3])

Total learnable parameters: 25784675


In [9]:
total_cnn_params > total_vit_params

True

In [10]:
total_cnn_params / total_vit_params

25.55101100436508

In [11]:
from gwtorch.modules.neural_net import SimpleCNN
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # Instantiate model
# class SimpleCNN(nn.Module):
#     def __init__(self, input_channels=3, dropout=0.2):
#         super(SimpleCNN, self).__init__()
#         self.conv1 = nn.Conv2d(input_channels, 16, kernel_size=3, stride=1, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
#         self.fc1 = nn.Linear(self._get_fc_input_size(input_channels), 32)  # <- need to calculate this
#         self.fc2 = nn.Linear(32, 3)
#         self.dropout = nn.Dropout(dropout)
    
#     def _get_fc_input_size(self, input_channels):
#         with torch.no_grad():
#             x = torch.zeros(1, input_channels, 224, 224)
#             x = F.max_pool2d(F.relu(self.conv1(x)), 2)
#             x = F.max_pool2d(F.relu(self.conv2(x)), 2)
#             return x.numel()

#     def forward(self, x):
#         x = F.relu(self.conv1(x))
#         x = F.max_pool2d(x, 2)   # 224 -> 112
#         x = F.relu(self.conv2(x))
#         x = F.max_pool2d(x, 2)   # 112 -> 56
#         x = x.view(x.size(0), -1)  # flatten
#         x = F.relu(self.fc1(x))
#         x = self.dropout(x)
#         x = self.fc2(x)
#         return x

model = SimpleCNN(input_channels=4)

# Print learnable parameters with their shapes
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.shape}")

# Count total learnable parameters
total_simple_cnn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_simple_cnn_params}")

conv1.weight: torch.Size([16, 4, 3, 3])
conv1.bias: torch.Size([16])
conv2.weight: torch.Size([32, 16, 3, 3])
conv2.bias: torch.Size([32])
fc1.weight: torch.Size([64, 100352])
fc1.bias: torch.Size([64])
fc2.weight: torch.Size([3, 64])
fc2.bias: torch.Size([3])

Total learnable parameters: 6428019


In [12]:
total_simple_cnn_params / total_vit_params

6.369767476428065

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, input_channels=3, num_classes=3, dropout=0.2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 224 → 112

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 112 → 56
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # → (B, 16, 1, 1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


In [18]:
model = SimpleCNN(input_channels=4)

total_simple_cnn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_simple_cnn_params}")


Total learnable parameters: 1515


In [20]:
total_vit_params / total_simple_cnn_params

335.5953795379538

In [16]:
class TinyCNN(nn.Module):
    def __init__(self, input_channels=3, num_classes=3):
        super().__init__()
        self.conv = nn.Conv2d(input_channels, 8, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(8, num_classes)

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = TinyCNN(input_channels=4)

total_simple_cnn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_simple_cnn_params}")


Total learnable parameters: 323
